# Semana 4 — HL7 FHIR R4: extraer por API REST, paginar y aplanar recursos

**Objetivo:** (A) consultar un servidor FHIR real por su API REST siguiendo la paginación del estándar, y (B) aplanar *Bundles* FHIR de Synthea a tablas analíticas, conciliándolas con la versión CSV de los mismos pacientes.

**Productos reales usados:** servidor público HAPI FHIR R4 (`hapi.fhir.org/baseR4`), `requests`, Bundles FHIR R4 de Synthea (noviembre 2021, mismos pacientes que el CSV).

**Al terminar tendrás:** `data/silver_fhir/*.parquet` y un reporte de conciliación FHIR vs CSV.

**Requisito:** semana 3 (usa `data/silver/`).

### Celda de configuración del entorno (se repite al inicio de cada semana)

**Explicación línea por línea**

- `import os`, `import sys`, `from pathlib import Path`: cargan tres módulos de la librería estándar. `os` lee variables de entorno, `sys` informa la versión de Python y `Path` representa rutas de archivos como objetos. **Por qué así:** `Path` evita concatenar rutas con texto (`"a" + "/" + "b"`), que falla entre sistemas operativos. **En la práctica real** todo código de datos moderno usa `pathlib`.
- `try: import google.colab` … `except ImportError`: intenta importar un módulo que solo existe en Colab. Si la importación funciona, `IN_COLAB = True`; si lanza `ImportError`, estamos en otra máquina. **Por qué importa:** el mismo notebook corre en Colab y en tu laptop sin editarlo. **Cuándo se usa:** siempre que un código deba ser portable entre entornos.
- `from google.colab import drive` y `drive.mount("/content/drive")`: conectan tu Google Drive al sistema de archivos de Colab. **Por qué:** el disco de Colab se borra al cerrar la sesión; en Drive los datos de la semana 1 siguen ahí el lunes siguiente. La primera vez Colab te pedirá autorizar el acceso.
- `PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")`: carpeta raíz del proyecto dentro de tu Drive.
- `Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()`: fuera de Colab usa la variable de entorno `HEALTHETL_PROJECT_DIR` si existe; si no, la carpeta actual (`"."`). `.resolve()` la convierte en ruta absoluta. **En la práctica real** la configuración se pasa por variables de entorno, nunca escrita a mano dentro del código (principio *12-factor app*).
- `DATA_DIR = PROJECT_DIR / "data"`: el operador `/` de `pathlib` une rutas de forma segura.
- `DATA_DIR.mkdir(parents=True, exist_ok=True)`: crea la carpeta y sus padres; `exist_ok=True` evita un error si ya existe. Esto hace la celda **idempotente**: puedes ejecutarla 1 o 10 veces con el mismo resultado, una propiedad central de todo pipeline ETL.
- Los tres `print`: confirman versión de Python, entorno y ruta. Registrar el entorno al inicio es un hábito de reproducibilidad.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # este módulo solo existe dentro de Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")
else:
    PROJECT_DIR = Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("¿Colab?:", IN_COLAB)
print("Carpeta de datos:", DATA_DIR)

## Parte A — API REST de un servidor FHIR

### Paso 1 — Una sesión HTTP robusta

**Explicación línea por línea**

- `import requests`: la librería HTTP más usada en Python.
- `from requests.adapters import HTTPAdapter` y `from urllib3.util.retry import Retry`: permiten reintentos automáticos.
- `FHIR_BASE = os.environ.get("FHIR_BASE", "https://hapi.fhir.org/baseR4")`: URL base del servidor; configurable por variable de entorno para apuntar mañana a otro servidor sin tocar el código.
- `session = requests.Session()`: una sesión reutiliza la conexión TCP entre llamadas (más rápido) y comparte encabezados.
- `session.headers.update({"Accept": "application/fhir+json"})`: pide respuestas en JSON FHIR, el tipo de contenido oficial del estándar.
- `Retry(total=3, backoff_factor=1.0, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"])`: hasta 3 reintentos, esperando cada vez más (1 s, 2 s, 4 s), solo ante "demasiadas solicitudes" (429) o errores del servidor (5xx), y solo en `GET` (leer es seguro de repetir; escribir no). **En la práctica real** los servidores públicos y de hospitales fallan de forma intermitente; sin reintentos con espera, un pipeline nocturno se cae por un error pasajero.
- `session.mount("https://", HTTPAdapter(max_retries=retry))`: aplica la política a toda URL `https://`.

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

FHIR_BASE = os.environ.get("FHIR_BASE", "https://hapi.fhir.org/baseR4")

session = requests.Session()
session.headers.update({"Accept": "application/fhir+json"})
retry = Retry(total=3, backoff_factor=1.0, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"])
session.mount("https://", HTTPAdapter(max_retries=retry))
print("Servidor:", FHIR_BASE)

### Paso 2 — Leer el CapabilityStatement (`/metadata`)

Todo servidor FHIR publica en `/metadata` qué versión del estándar implementa y qué recursos y búsquedas soporta. Es lo primero que se consulta antes de integrar.

**Explicación línea por línea**

- `try: ... except requests.RequestException as exc:` captura cualquier error de red; el servidor público puede estar caído (es un servidor de pruebas sin garantía de disponibilidad).
- `r = session.get(f"{FHIR_BASE}/metadata", timeout=30)`: `timeout=30` evita esperar indefinidamente. **Regla de práctica real:** toda llamada HTTP lleva *timeout*.
- `r.raise_for_status()`: convierte un código 4xx/5xx en excepción.
- `cap = r.json()`: interpreta el cuerpo como JSON → diccionario de Python.
- `cap.get("fhirVersion")`: versión, por ejemplo `4.0.1` (= R4).
- `[res["type"] for res in cap["rest"][0]["resource"]][:15]`: lista de tipos de recurso soportados.
- `LIVE = True/False`: bandera que decide si la Parte A corre en vivo. La Parte B funciona siempre, sin internet al servidor FHIR.

In [ ]:
try:
    r = session.get(f"{FHIR_BASE}/metadata", timeout=30)
    r.raise_for_status()
    cap = r.json()
    print("FHIR version:", cap.get("fhirVersion"))
    print("Software:", cap.get("software", {}).get("name"), cap.get("software", {}).get("version"))
    print("Recursos (15 primeros):", [res["type"] for res in cap["rest"][0]["resource"]][:15])
    LIVE = True
except requests.RequestException as exc:
    print("Servidor FHIR no disponible ahora:", type(exc).__name__)
    LIVE = False

### Paso 3 — Búsqueda con paginación siguiendo `Bundle.link[next]`

Una búsqueda FHIR devuelve un `Bundle` de tipo `searchset` con una página de resultados y enlaces de navegación. La URL de la página siguiente la decide el servidor y es **opaca**: el cliente debe seguir el enlace `next` tal cual, no construirlo.

**Explicación línea por línea**

- `def fetch_all(url, params=None, max_pages=5):` descarga hasta `max_pages` páginas. El límite protege al servidor público y a tu sesión.
- `resources, pages = [], 0`: acumuladores.
- `while url and pages < max_pages:` sigue mientras exista página siguiente y no superes el límite.
- `r = session.get(url, params=params, timeout=60)`: en la primera llamada `params` lleva la búsqueda (`_count=50`); `requests` los codifica en la URL.
- `for e in bundle.get("entry", [])`: cada `entry` trae un `resource`. `.get(..., [])` cubre páginas vacías.
- `if e.get("search", {}).get("mode", "match") == "match"`: conserva solo resultados que cumplen la búsqueda; excluye recursos añadidos con `_include` (modo `include`).
- `url = next((l["url"] for l in bundle.get("link", []) if l["relation"] == "next"), None)`: busca el enlace `next`; si no hay, `None` termina el ciclo. `next(generador, valor_por_defecto)` toma el primer elemento o el valor por defecto.
- `params = None`: el enlace `next` ya contiene toda la consulta; reenviar `params` la duplicaría.

In [ ]:
def fetch_all(url, params=None, max_pages=5):
    resources, pages = [], 0
    while url and pages < max_pages:
        r = session.get(url, params=params, timeout=60)
        r.raise_for_status()
        bundle = r.json()
        for e in bundle.get("entry", []):
            if e.get("search", {}).get("mode", "match") == "match":
                resources.append(e["resource"])
        url = next((l["url"] for l in bundle.get("link", []) if l["relation"] == "next"), None)
        params = None
        pages += 1
    return resources


if LIVE:
    live_patients = fetch_all(f"{FHIR_BASE}/Patient", params={"_count": 50}, max_pages=3)
    print("Patient descargados del servidor público:", len(live_patients))
else:
    live_patients = []
    print("Parte A omitida: sigue con la Parte B")

### Paso 4 — Perfil rápido de datos de un servidor público (calidad del mundo real)

El servidor HAPI público lo usan desarrolladores de todo el mundo para pruebas: sus datos son heterogéneos e incompletos, más parecidos a datos reales que Synthea.

**Explicación línea por línea**

- `pd.DataFrame([...])`: construye una tabla con una fila por paciente.
- `p.get("gender")`, `p.get("birthDate")`: en FHIR casi todo es opcional; `.get` devuelve `None` si falta, en vez de romper el código como `p["gender"]`.
- `bool(p.get("name"))`: verdadero si hay al menos un nombre.
- `df_live.isna().mean().round(2)`: proporción de ausentes por columna.
- `df_live["gender"].value_counts(dropna=False)`: dominio observado; según FHIR R4 el dominio permitido de `gender` es `male`, `female`, `other`, `unknown`.

In [ ]:
import pandas as pd

if live_patients:
    df_live = pd.DataFrame([{
        "id": p.get("id"),
        "gender": p.get("gender"),
        "birthDate": p.get("birthDate"),
        "has_name": bool(p.get("name")),
    } for p in live_patients])
    print(df_live.isna().mean().round(2))
    print(df_live["gender"].value_counts(dropna=False))

## Parte B — Aplanar Bundles FHIR de Synthea

### Paso 5 — Descargar los Bundles (mismos pacientes que el CSV)

**Explicación línea por línea**

- `FHIR_ZIP_URL`: ZIP oficial de Synthea con un Bundle FHIR R4 por paciente, generado en la misma corrida que el CSV de noviembre 2021.
- Descarga idempotente igual que en la semana 1. Son ~95 MB comprimidos y ~1,1 GB descomprimidos, por eso **no** descomprimimos: leeremos cada archivo directamente desde el ZIP.
- `bundle_names = [n for n in zf.namelist() if ... "hospitalInformation" not in n ...]`: cada paciente es un archivo `Nombre_Apellido_UUID.json`. Excluimos dos archivos especiales que traen organizaciones y profesionales, no pacientes.

In [ ]:
import json
import urllib.request
import zipfile

FHIR_ZIP_URL = (
    "https://raw.githubusercontent.com/synthetichealth/synthea-sample-data/"
    "main/downloads/synthea_sample_data_fhir_r4_nov2021.zip"
)
FHIR_ZIP = DATA_DIR / "raw" / "synthea_sample_data_fhir_r4_nov2021.zip"
if not FHIR_ZIP.exists():
    print("Descargando ~95 MB...")
    urllib.request.urlretrieve(FHIR_ZIP_URL, FHIR_ZIP)

zf = zipfile.ZipFile(FHIR_ZIP)
bundle_names = [
    n for n in zf.namelist()
    if n.endswith(".json") and "hospitalInformation" not in n and "practitionerInformation" not in n
]
print("Bundles de pacientes:", len(bundle_names))

### Paso 6 — Anatomía de un Bundle

**Explicación línea por línea**

- `first = json.loads(zf.read(bundle_names[0]))`: `zf.read` devuelve los bytes del archivo dentro del ZIP; `json.loads` los convierte en diccionario.
- `first["resourceType"], first["type"]`: es un `Bundle` de tipo `transaction` (pensado para cargarse de una vez en un servidor), no `searchset`.
- `Counter(e["resource"]["resourceType"] for e in first["entry"])`: cuenta recursos por tipo dentro de un paciente: `Encounter`, `Observation`, `Condition`, `Claim`...
- `obs_bp = next(...)`: busca la primera observación de presión arterial (LOINC 85354-9). Verás que **no** tiene `valueQuantity`: la sistólica y la diastólica viven dentro de `component`. **Por qué importa:** si aplanas solo `valueQuantity`, pierdes toda la presión arterial. Es el patrón de *panel* de FHIR.
- `json.dumps(obs_bp["component"], indent=1)[:600]`: muestra los componentes (corta a 600 caracteres).

In [ ]:
from collections import Counter

first = json.loads(zf.read(bundle_names[0]))
print(first["resourceType"], first["type"], "| entradas:", len(first["entry"]))
print(Counter(e["resource"]["resourceType"] for e in first["entry"]).most_common(8))

obs_bp = next(
    e["resource"] for e in first["entry"]
    if e["resource"]["resourceType"] == "Observation"
    and e["resource"]["code"]["coding"][0]["code"] == "85354-9"
)
print("¿valueQuantity?:", "valueQuantity" in obs_bp)
print(json.dumps(obs_bp["component"], indent=1)[:600])

### Paso 7 — Funciones auxiliares para navegar FHIR

**Explicación línea por línea**

- `def ref_id(ref):` extrae el identificador de una referencia. En un Bundle de transacción las referencias son `urn:uuid:<id>`; en un servidor son `Patient/<id>`. `ref.split(":")[-1].split("/")[-1]` cubre ambas formas. `if not ref: return None` cubre referencias ausentes.
- `def coding(cc, system=None):` devuelve el primer `Coding` de un `CodeableConcept` (opcionalmente de un sistema concreto, p. ej. `http://snomed.info/sct`). Un `CodeableConcept` puede traer varios códigos de distintos sistemas para el mismo concepto.
- `(cc or {}).get("coding", [])`: tolera `cc = None`.
- `def ext_value(resource, url):` busca una extensión por su URL. Las extensiones son el mecanismo de FHIR para añadir datos que el recurso base no define (raza y etnicidad en US Core; en AU Core, por ejemplo, el indicador de origen indígena).

In [ ]:
SNOMED = "http://snomed.info/sct"
LOINC = "http://loinc.org"


def ref_id(ref):
    if not ref:
        return None
    return ref.split(":")[-1].split("/")[-1]


def coding(cc, system=None):
    for c in (cc or {}).get("coding", []):
        if system is None or c.get("system") == system:
            return c
    return {}


def ext_value(resource, url):
    for ext in resource.get("extension", []):
        if ext.get("url") == url:
            return ext
    return None


print(ref_id("urn:uuid:abc-123"), ref_id("Patient/abc-123"), coding(obs_bp["code"], LOINC).get("code"))

### Paso 8 — Aplanadores por tipo de recurso

**Explicación línea por línea**

- `def flat_patient(r):` toma lo esencial de `Patient`: `id`, `gender`, `birthDate`, `deceasedDateTime`.
- `race = ext_value(r, ".../us-core-race")`: extensión compleja; dentro tiene sub-extensiones. `next((e["valueCoding"]["display"] for e in race["extension"] if e["url"] == "ombCategory"), None)` toma la categoría OMB. No guardamos nombre, dirección ni teléfono: **minimización** también al aplanar.
- `def flat_encounter(r):` `r["class"]["code"]` es el tipo de encuentro (`AMB`, `EMER`, `IMP`... del vocabulario HL7 ActCode); `r["period"]` trae inicio y fin.
- `def flat_condition(r):` código SNOMED CT, estado clínico (`active`, `resolved`) y fechas de inicio (`onsetDateTime`) y fin (`abatementDateTime`).
- `def flat_observations(r):` devuelve una **lista** de filas porque una observación con componentes genera varias. Si hay `valueQuantity`, una fila; si hay `valueCodeableConcept` (p. ej. tabaquismo), guarda el texto; si hay `component`, una fila por componente, conservando `parent_code` (el panel) para el linaje.
- `"value_num": vq.get("value")`, `"unit": vq.get("code") or vq.get("unit")`: el código UCUM es lo estandarizado; `unit` es texto libre de respaldo.

In [ ]:
RACE_URL = "http://hl7.org/fhir/us/core/StructureDefinition/us-core-race"


def flat_patient(r):
    race = ext_value(r, RACE_URL)
    race_display = None
    if race:
        race_display = next((e["valueCoding"]["display"] for e in race.get("extension", []) if e["url"] == "ombCategory"), None)
    return {"patient_id": r["id"], "gender": r.get("gender"), "birth_date": r.get("birthDate"),
            "deceased_ts": r.get("deceasedDateTime"), "race": race_display}


def flat_encounter(r):
    return {"encounter_id": r["id"], "patient_id": ref_id(r.get("subject", {}).get("reference")),
            "class_code": r.get("class", {}).get("code"),
            "type_snomed": coding(r.get("type", [{}])[0], SNOMED).get("code"),
            "start_ts": r.get("period", {}).get("start"), "end_ts": r.get("period", {}).get("end")}


def flat_condition(r):
    c = coding(r.get("code"), SNOMED)
    return {"condition_id": r["id"], "patient_id": ref_id(r.get("subject", {}).get("reference")),
            "encounter_id": ref_id(r.get("encounter", {}).get("reference")),
            "snomed_code": c.get("code"), "display": c.get("display"),
            "clinical_status": coding(r.get("clinicalStatus")).get("code"),
            "onset_ts": r.get("onsetDateTime"), "abatement_ts": r.get("abatementDateTime")}


def flat_observations(r):
    base = {"observation_id": r["id"], "patient_id": ref_id(r.get("subject", {}).get("reference")),
            "encounter_id": ref_id(r.get("encounter", {}).get("reference")),
            "effective_ts": r.get("effectiveDateTime"),
            "category": coding((r.get("category") or [{}])[0]).get("code")}
    top = coding(r.get("code"), LOINC)
    rows = []
    if "valueQuantity" in r:
        vq = r["valueQuantity"]
        rows.append({**base, "parent_code": None, "loinc_code": top.get("code"),
                     "value_num": vq.get("value"), "unit": vq.get("code") or vq.get("unit"), "value_text": None})
    elif "valueCodeableConcept" in r:
        rows.append({**base, "parent_code": None, "loinc_code": top.get("code"), "value_num": None,
                     "unit": None, "value_text": coding(r["valueCodeableConcept"]).get("display")})
    for comp in r.get("component", []):
        vq = comp.get("valueQuantity", {})
        rows.append({**base, "parent_code": top.get("code"), "loinc_code": coding(comp.get("code"), LOINC).get("code"),
                     "value_num": vq.get("value"), "unit": vq.get("code") or vq.get("unit"), "value_text": None})
    return rows


flat_observations(obs_bp)

### Paso 9 — Recorrer los 555 Bundles y construir las tablas

**Explicación línea por línea**

- `rows = {"patient": [], ...}`: un acumulador por tabla de destino.
- `for i, name in enumerate(bundle_names, start=1):` recorre los archivos numerándolos desde 1.
- `bundle = json.loads(zf.read(name))`: lee y parsea cada Bundle **uno a la vez**; nunca hay más de un paciente en memoria como JSON. **En la práctica real** este patrón de *streaming* es lo que permite procesar exportaciones *Bulk FHIR* de millones de recursos.
- `rt = res["resourceType"]` y el `if/elif`: despacha cada recurso a su aplanador. Los demás tipos (`Claim`, `ExplanationOfBenefit`...) se ignoran de forma explícita.
- `rows["observation"].extend(flat_observations(res))`: `extend` añade varias filas de una vez.
- `if i % 100 == 0: print(...)`: progreso cada 100 archivos.
- `tables = {k: pd.DataFrame(v) for k, v in rows.items()}`: convierte cada lista en DataFrame.

In [ ]:
import time

rows = {"patient": [], "encounter": [], "condition": [], "observation": []}
t0 = time.time()
for i, name in enumerate(bundle_names, start=1):
    bundle = json.loads(zf.read(name))
    for entry in bundle.get("entry", []):
        res = entry["resource"]
        rt = res["resourceType"]
        if rt == "Patient":
            rows["patient"].append(flat_patient(res))
        elif rt == "Encounter":
            rows["encounter"].append(flat_encounter(res))
        elif rt == "Condition":
            rows["condition"].append(flat_condition(res))
        elif rt == "Observation":
            rows["observation"].extend(flat_observations(res))
    if i % 100 == 0:
        print(f"{i} bundles procesados ({time.time() - t0:.0f} s)")

tables = {k: pd.DataFrame(v) for k, v in rows.items()}
for k, df in tables.items():
    print(f"{k:<12} {len(df):>9,} filas")

### Paso 10 — Tipos y publicación en `silver_fhir`

**Explicación línea por línea**

- `for col in [...]: df[col] = pd.to_datetime(df[col], utc=True, format="ISO8601", errors="coerce")`: FHIR usa ISO 8601 con zona horaria local del paciente (por ejemplo `-05:00`); `utc=True` normaliza todo a UTC, igual que hicimos en silver con el CSV.
- `.dt.tz_localize(None)`: quita la etiqueta de zona después de convertir a UTC, para que las fechas sean comparables con las de `data/silver/` (que guardamos como UTC sin zona).
- `if col in df.columns`: protección por si una tabla viniera vacía.
- `df.to_parquet(OUT_DIR / f"{k}.parquet", index=False)`: publica cada tabla.

In [ ]:
OUT_DIR = DATA_DIR / "silver_fhir"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATE_COLS = {"patient": ["birth_date", "deceased_ts"], "encounter": ["start_ts", "end_ts"],
             "condition": ["onset_ts", "abatement_ts"], "observation": ["effective_ts"]}
for k, df in tables.items():
    for col in DATE_COLS[k]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], utc=True, format="ISO8601", errors="coerce").dt.tz_localize(None)
    df.to_parquet(OUT_DIR / f"{k}.parquet", index=False)
tables["observation"].head()

### Paso 11 — Conciliación FHIR vs CSV: ¿es la misma información?

Dos representaciones de los mismos pacientes deben contar la misma historia. Conciliar es cómo un equipo de datos demuestra que un nuevo canal (FHIR) puede reemplazar a uno antiguo (CSV).

**Explicación línea por línea**

- `csv_p = pd.read_parquet(DATA_DIR / "silver" / "patients.parquet")`: pacientes de la semana 3.
- `cobertura = tables["patient"]["patient_id"].isin(csv_p["patient_id"]).mean()`: proporción de pacientes FHIR que existen en el CSV (esperado: 100 %).
- `len(fhir_ids) / len(csv_p)`: qué fracción del CSV está en FHIR. Verás que el ZIP FHIR trae menos pacientes que el CSV: **hallazgo de cobertura** que debes documentar antes de usar FHIR como fuente única.
- `def per_patient(df, id_col)`: cuenta registros por paciente.
- `csv_c = csv_c[csv_c["patient_id"].isin(fhir_ids)]`: compara solo pacientes presentes en ambos.
- `comp = pd.concat([...], axis=1).fillna(0)`: une los conteos lado a lado; `fillna(0)` para pacientes con 0 en un lado.
- `(comp["fhir"] == comp["csv"]).mean()`: proporción de pacientes con el mismo número de condiciones.
- Para presión sistólica se compara el número de valores LOINC 8480-6 por paciente: en FHIR salen de `component`, en CSV de filas separadas. Si coinciden, tu aplanador de componentes es correcto.

In [ ]:
csv_p = pd.read_parquet(DATA_DIR / "silver" / "patients.parquet")
fhir_ids = set(tables["patient"]["patient_id"])
print(f"Pacientes FHIR presentes en CSV: {tables['patient']['patient_id'].isin(csv_p['patient_id']).mean():.0%}")
print(f"Fracción del CSV cubierta por FHIR: {len(fhir_ids) / len(csv_p):.0%}")


def per_patient(df, id_col="patient_id"):
    return df.groupby(id_col).size()


csv_c = pd.read_parquet(DATA_DIR / "silver" / "conditions.parquet")
csv_c = csv_c[csv_c["patient_id"].isin(fhir_ids)]
comp = pd.concat([per_patient(tables["condition"]).rename("fhir"), per_patient(csv_c).rename("csv")], axis=1).fillna(0)
print(f"Pacientes con igual nº de condiciones: {(comp['fhir'] == comp['csv']).mean():.1%}")

csv_o = pd.read_parquet(DATA_DIR / "silver" / "observations.parquet", columns=["patient_id", "loinc_code"])
csv_sbp = csv_o[(csv_o["loinc_code"] == "8480-6") & csv_o["patient_id"].isin(fhir_ids)]
fhir_sbp = tables["observation"][tables["observation"]["loinc_code"] == "8480-6"]
comp_sbp = pd.concat([per_patient(fhir_sbp).rename("fhir"), per_patient(csv_sbp).rename("csv")], axis=1).fillna(0)
print(f"Pacientes con igual nº de sistólicas: {(comp_sbp['fhir'] == comp_sbp['csv']).mean():.1%}")
comp[comp["fhir"] != comp["csv"]].head()

### Autoevaluación

1. ¿Por qué nunca debes construir tú la URL de la página siguiente en una búsqueda FHIR?
2. ¿Qué perderías si tu aplanador ignorara `Observation.component`?
3. ¿Por qué `ref_id` maneja tanto `urn:uuid:` como `Patient/`?
4. El ZIP FHIR cubre menos pacientes que el CSV. ¿Qué harías antes de sustituir la fuente CSV por FHIR en producción?

**Commit sugerido:** `feat(s04): extracción FHIR por API con paginación y aplanado de Bundles con conciliación`